# Parte 2 – Evaluación, comparación de modelos y extracción estructurada

**Pregunta guía:** ¿podemos convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?
Aquí medimos **cuánto se equivocan** los modelos y elegimos cuál usaríamos en un entorno real.

**Reglas que seguimos para que la comparación sea justa** (definidas antes de ver cualquier métrica):

1. **Muestra fija:** 10 documentos y 10 clips del corpus de la Parte 1, elegidos por ser cortos (para poder transcribirlos a mano) e incluir todos los tipos y condiciones.
   No se agregaron datos nuevos.
2. **Referencias manuales:** cada documento y clip de la muestra fue transcrito a mano, sin mirar la salida de ningún modelo.
3. **Mismos archivos para todos los modelos**, con la configuración fijada de antemano y sin ajustarla según los resultados.
4. **Misma normalización** para referencias y salidas: minúsculas, sin puntuación ni espacios repetidos, conservando las tildes (`voz.normalizar`).
   En el WER se eliminan las marcas `[HABLANTE_n]`, que no son palabras habladas.


In [1]:
import json
import os
import re
import time
import warnings
from pathlib import Path

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import soundfile as sf
import torch

import voz

pd.set_option("display.max_colwidth", 90)
DATOS = Path("datos")
RAW = DATOS / "raw"
REFERENCIAS = RAW / "referencias"
EVAL = RAW / "evaluacion"
EVAL.mkdir(parents=True, exist_ok=True)
DISPOSITIVO = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

documentos = pd.read_csv(DATOS / "documentos.csv", keep_default_na=False)
transcripciones = pd.read_csv(DATOS / "transcripciones.csv", keep_default_na=False)
print("documentos:", documentos.shape, "| transcripciones:", transcripciones.shape, "| dispositivo:", DISPOSITIVO)

documentos: (25, 8) | transcripciones: (28, 10) | dispositivo: mps


## 1 · Muestra de evaluación y referencias manuales

| | Muestra |
|---|---|
| **10 documentos** | 3 impresos (`doc_001`, `doc_007`, `doc_008`), 4 fotos (`doc_009`, `doc_014`, `doc_015`, `doc_016`), 3 manuscritos (`doc_022`, `doc_023`, `doc_025`) |
| **10 clips** | 4 frases leídas de OpenSLR 73 (`clip_024`–`clip_027`) y 6 entrevistas de Commons en distintas condiciones: calle (`clip_003`, `clip_022`), feria (`clip_015`) y fiesta patronal con mucho ruido (`clip_005`, `clip_006`, `clip_007`) |

Las transcripciones manuales están en `datos/raw/referencias/<id>.txt` y se copian a la columna `texto_referencia` de los CSV.

In [2]:
MUESTRA_DOCS = ["doc_001", "doc_007", "doc_008", "doc_009", "doc_014", "doc_015", "doc_016", "doc_022", "doc_023", "doc_025"]
MUESTRA_CLIPS = ["clip_024", "clip_025", "clip_026", "clip_027", "clip_003", "clip_022", "clip_015", "clip_005", "clip_006", "clip_007"]


def referencia(id_):
    "Transcripción manual de un documento o clip (None si todavía no existe o está vacía)."
    ruta = REFERENCIAS / f"{id_}.txt"
    texto = ruta.read_text(encoding="utf-8").strip() if ruta.exists() else ""
    return texto or None


SEUDONIMOS = json.loads((RAW / "seudonimos.json").read_text(encoding="utf-8"))
COMPLETOS = {n: p for n, p in SEUDONIMOS.items() if len(n.split()) > 1 or len(n) > 10   # nombre y apellido que el ASR unió en una palabra}
SUELTOS = {n: p for n, p in SEUDONIMOS.items() if n not in COMPLETOS}


def seudonimizar(texto):
    "El mismo reemplazo de nombres de la Parte 1 (nombres completos al 80 %, sueltos al 90 %)."
    return voz.seudonimizar(voz.seudonimizar(texto, COMPLETOS, parecido=80), SUELTOS, parecido=90)


# texto_referencia en los CSV: documentos tal cual (solo personajes históricos); audios seudonimizados como el ASR
documentos["texto_referencia"] = [referencia(i) or "" for i in documentos["id"]]
transcripciones["texto_referencia"] = [seudonimizar(referencia(i)) if referencia(i) else "" for i in transcripciones["id"]]
documentos.to_csv(DATOS / "documentos.csv", index=False)
transcripciones.to_csv(DATOS / "transcripciones.csv", index=False)

muestra_docs = documentos.set_index("id").loc[MUESTRA_DOCS, ["tipo", "archivo", "texto_ocr"]].reset_index()
muestra_docs["referencia"] = [referencia(i) for i in MUESTRA_DOCS]
muestra_clips = transcripciones.set_index("id").loc[MUESTRA_CLIPS, ["condicion", "archivo_audio", "duracion_seg", "n_hablantes"]].reset_index()
muestra_clips["referencia"] = [seudonimizar(referencia(i)) if referencia(i) else None for i in MUESTRA_CLIPS]

print(f"Referencias listas: {muestra_docs['referencia'].notna().sum()}/10 documentos, "
      f"{muestra_clips['referencia'].notna().sum()}/10 clips | audio de la muestra: {muestra_clips['duracion_seg'].sum() / 60:.1f} min")
pd.concat([muestra_docs[["id", "tipo"]].rename(columns={"tipo": "tipo / condición"}),
           muestra_clips[["id", "condicion"]].rename(columns={"condicion": "tipo / condición"})])

Referencias listas: 10/10 documentos, 10/10 clips | audio de la muestra: 7.5 min


,id,tipo / condición
0,doc_001,impreso
1,doc_007,impreso
2,doc_008,impreso
3,doc_009,foto
4,doc_014,foto
5,doc_015,foto
6,doc_016,foto
7,doc_022,manuscrito
8,doc_023,manuscrito
9,doc_025,manuscrito


## 2 · OCR: PaddleOCR (línea base) frente a PaddleOCR-VL

- **Línea base – PP-OCRv6 *medium*:** la misma lectura de la Parte 1 (PaddleOCR 3.7, sin preprocesamiento ni ajustes). Se reutiliza su texto y su tiempo.
- **Alternativa – PaddleOCR-VL 1.6:** un modelo de visión y lenguaje (0,9 B parámetros) que primero detecta los bloques de la página
  (PP-DocLayoutV3) y luego "lee" cada bloque generando texto, como un modelo de lenguaje. Lo vimos en la Sesión 10.
  **Por qué este y no Tesseract:** Tesseract no está disponible en la Mac de trabajo (no hay Homebrew) y es un OCR clásico parecido a la línea base;
  PaddleOCR-VL es un enfoque distinto, que en la clase leyó mejor la letra manuscrita. Configuración: la de la clase, con `max_new_tokens=2048`
  para no cortar los documentos más largos de la muestra.


In [3]:
from paddleocr import PaddleOCRVL

DIR_VL = EVAL / "ocr_paddleocr_vl"
DIR_VL.mkdir(exist_ok=True)
vlm = None


def leer_vlm(ruta, max_tokens=2048):
    "Lee un documento con PaddleOCR-VL (función de la Sesión 10). Devuelve el texto y los segundos que tardó."
    t0 = time.perf_counter()
    r = vlm.predict(str(ruta), max_new_tokens=max_tokens)[0]
    texto = r.markdown["markdown_texts"]
    texto = re.sub(r"<[^>]+>", " ", texto)                  # quitar etiquetas HTML de imágenes incrustadas
    texto = re.sub(r"^#+\s*", "", texto, flags=re.M)        # quitar las marcas de título de Markdown
    return texto.strip(), time.perf_counter() - t0


def ocr_vl(id_, ruta):
    "Lectura con PaddleOCR-VL, con respaldo en disco para no repetirla."
    global vlm
    crudo = DIR_VL / f"{id_}.json"
    if crudo.exists():
        return json.loads(crudo.read_text(encoding="utf-8"))
    if vlm is None:
        vlm = PaddleOCRVL(engine="transformers", use_doc_orientation_classify=False, use_doc_unwarping=False)
        for red in voz_redes(vlm):
            red.to(DISPOSITIVO)
        leer_vlm(ruta)                                    # calentamiento: no se cuenta
    texto, segundos = leer_vlm(ruta)
    salida = {"id": id_, "texto": texto, "segundos": round(segundos, 2)}
    crudo.write_text(json.dumps(salida, ensure_ascii=False, indent=1), encoding="utf-8")
    return salida


def voz_redes(obj, _prof=0, _vistos=None):
    "Redes de PyTorch dentro de un modelo de PaddleOCR (función 'redes' de la Sesión 10)."
    _vistos = set() if _vistos is None else _vistos
    if id(obj) in _vistos or _prof > 7:
        return
    _vistos.add(id(obj))
    if isinstance(obj, torch.nn.Module):
        yield obj
        return
    for hijo in (vars(obj).values() if hasattr(obj, "__dict__") else []):
        for x in (hijo.values() if isinstance(hijo, dict) else [hijo]):
            if not isinstance(x, (str, int, float, bool, type(None), list, tuple, set)):
                yield from voz_redes(x, _prof + 1, _vistos)


base_seg = {i: json.loads((RAW / "documentos" / "ocr" / f"{i}.json").read_text(encoding="utf-8"))["segundos"] for i in MUESTRA_DOCS}
salidas_ocr = []
for id_, ruta, texto_base in zip(muestra_docs["id"], muestra_docs["archivo"], muestra_docs["texto_ocr"]):
    vl = ocr_vl(id_, ruta)
    salidas_ocr += [{"id": id_, "modelo": "PP-OCRv6 (base)", "texto": texto_base, "segundos": base_seg[id_]},
                    {"id": id_, "modelo": "PaddleOCR-VL 1.6", "texto": vl["texto"], "segundos": vl["segundos"]}]
salidas_ocr = pd.DataFrame(salidas_ocr).merge(muestra_docs[["id", "tipo"]], on="id")
salidas_ocr.pivot_table(index="modelo", values="segundos", aggfunc=["mean", "sum"]).round(2)

,mean,sum
,segundos,segundos
modelo,,
PP-OCRv6 (base),3.04,30.41
PaddleOCR-VL 1.6,5.46,54.59


## 3 · ASR: faster-whisper *small* (línea base) frente a Whisper *turbo* y Qwen3-ASR

Tres modelos sobre los mismos 10 clips, con el mismo tratamiento del audio:

| Modelo | Papel | Dónde corre |
|---|---|---|
| faster-whisper **small** (int8) | línea base que pide el enunciado | CPU |
| faster-whisper **large-v3-turbo** (int8) | el modelo usado en la Parte 1 | CPU |
| **Qwen3-ASR 0.6B** (Alibaba, Apache 2.0) | ASR alternativo, de otra familia | GPU de la Mac (MPS) |

- Idioma fijado en `es`, VAD activado (pausas de 0,3 s) y la misma **regla de respaldo** de la Parte 1:
  si el VAD reconoce voz en menos del 30 % del clip, se transcribe sin VAD (Whisper) o en ventanas fijas de 12 s (Qwen3-ASR).
- **Latencia:** se mide el tiempo de transcripción de cada clip (sin contar la carga del modelo) y se calcula el
  **factor de tiempo real** = segundos de proceso / segundos de audio. Menos de 1 significa más rápido que el audio.


In [4]:
from faster_whisper import WhisperModel

DIR_ASR = EVAL / "asr"
DIR_ASR.mkdir(exist_ok=True)
VOZ_MINIMA = 0.30


def con_whisper(modelo):
    def transcribir(audio):
        segs, _ = modelo.transcribe(audio, language="es", beam_size=5, vad_filter=True,
                                    vad_parameters={"min_silence_duration_ms": 300})
        segs = [s for s in segs if s.text.strip()]
        if sum(s.end - s.start for s in segs) < VOZ_MINIMA * len(audio) / voz.SR:      # regla de respaldo
            segs, _ = modelo.transcribe(audio, language="es", beam_size=5, vad_filter=False)
        return " ".join(s.text.strip() for s in segs)
    return transcribir


def con_qwen(t):
    def transcribir(audio):
        segs = t.segmentos(audio, idioma="es")
        if sum(s.fin - s.inicio for s in segs) < VOZ_MINIMA * len(audio) / voz.SR:     # regla de respaldo
            paso = 12 * voz.SR
            return " ".join(filter(None, (t.transcribir(audio[i:i + paso], "es") for i in range(0, len(audio), paso))))
        return " ".join(s.texto for s in segs)
    return transcribir


MODELOS_ASR = {
    "faster-whisper small (base)": lambda: con_whisper(WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=8)),
    "faster-whisper large-v3-turbo": lambda: con_whisper(voz.Transcriptor("faster-whisper").modelo),
    "Qwen3-ASR 0.6B": lambda: con_qwen(voz.Transcriptor("qwen3-asr-0.6b", device=DISPOSITIVO)),
}

audios = {i: sf.read(r, dtype="float32")[0] for i, r in zip(muestra_clips["id"], muestra_clips["archivo_audio"])}
salidas_asr = []
for nombre, cargar in MODELOS_ASR.items():
    clave = re.sub(r"\W+", "_", nombre.lower()).strip("_")
    respaldo = DIR_ASR / f"{clave}.json"
    resultado = json.loads(respaldo.read_text(encoding="utf-8")) if respaldo.exists() else {}
    if len(resultado) < len(audios):
        transcribir = cargar()
        transcribir(audios["clip_024"])                    # calentamiento: no se cuenta
        for id_, audio in audios.items():
            if id_ not in resultado:
                t0 = time.perf_counter()
                texto = transcribir(audio)
                resultado[id_] = {"texto": texto, "segundos": round(time.perf_counter() - t0, 2)}
        respaldo.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
        del transcribir
    salidas_asr += [{"id": i, "modelo": nombre, **r} for i, r in resultado.items()]

salidas_asr = pd.DataFrame(salidas_asr).merge(muestra_clips[["id", "condicion", "duracion_seg"]], on="id")
latencia = (salidas_asr.groupby("modelo")[["segundos", "duracion_seg"]].sum()
            .assign(factor_tiempo_real=lambda d: (d["segundos"] / d["duracion_seg"]).round(3)))
latencia

,segundos,duracion_seg,factor_tiempo_real
modelo,,,
Qwen3-ASR 0.6B,43.10,450.5,0.096
faster-whisper large-v3-turbo,121.64,450.5,0.270
faster-whisper small (base),67.34,450.5,0.149


## 4 · Métricas: CER (documentos) y WER (audio)

- **CER** (tasa de error por carácter) = (sustituciones + borrados + inserciones) / caracteres de la referencia.
- **WER** (tasa de error por palabra): lo mismo, contando palabras.
- Ambas se calculan con `voz.cer` y `voz.wer` (jiwer), después de normalizar referencia y salida igual.
  Se reporta el valor **global** de cada modelo (sumando errores de toda la muestra) y el de cada tipo o condición.


In [5]:
def sin_marcas(texto):
    return re.sub(r"\[HABLANTE_\d+\]", " ", texto)


def metricas(salidas, refs, medida, grupo):
    con_ref = salidas[salidas["id"].isin(refs.dropna().index)]
    if con_ref.empty:
        return None
    f = voz.cer if medida == "CER" else voz.wer
    filas = []
    for (modelo, g), parte in con_ref.groupby(["modelo", grupo]):
        filas.append({"modelo": modelo, grupo: g, medida: f([refs[i] for i in parte["id"]], [sin_marcas(t) for t in parte["texto"]])})
    for modelo, parte in con_ref.groupby("modelo"):
        filas.append({"modelo": modelo, grupo: "GLOBAL", medida: f([refs[i] for i in parte["id"]], [sin_marcas(t) for t in parte["texto"]])})
    return pd.DataFrame(filas).pivot(index="modelo", columns=grupo, values=medida).round(3)


refs_docs = muestra_docs.set_index("id")["referencia"]
refs_clips = muestra_clips.set_index("id")["referencia"]
cer_tabla = metricas(salidas_ocr, refs_docs, "CER", "tipo")
wer_tabla = metricas(salidas_asr, refs_clips, "WER", "condicion")
faltan = (10 - refs_docs.notna().sum(), 10 - refs_clips.notna().sum())
if faltan != (0, 0):
    print(f"⚠ Faltan referencias: {faltan[0]} documentos y {faltan[1]} clips. Las métricas son parciales.")
print("CER por modelo y tipo de documento:")
display(cer_tabla)
print("WER por modelo y condición de grabación:")
display(wer_tabla)

CER por modelo y tipo de documento:


tipo,GLOBAL,foto,impreso,manuscrito
modelo,,,,
PP-OCRv6 (base),0.111,0.101,0.036,0.364
PaddleOCR-VL 1.6,0.096,0.104,0.041,0.258


WER por modelo y condición de grabación:


condicion,GLOBAL,calle,feria,fiesta patronal,lectura limpia
modelo,,,,,
Qwen3-ASR 0.6B,0.302,0.197,0.320,0.361,0.000
faster-whisper large-v3-turbo,0.236,0.184,0.338,0.236,0.000
faster-whisper small (base),0.349,0.262,0.316,0.421,0.043


### CER por documento

El error de cada modelo en cada documento de la muestra: muestra **dónde** falla cada uno.

In [6]:
def error_por_item(salidas, refs, f):
    filas = []
    for r in salidas.itertuples():
        ref = refs.get(r.id)
        filas.append({"id": r.id, "modelo": r.modelo, "error": round(f([ref], [sin_marcas(r.texto)]), 3) if ref else None})
    return pd.DataFrame(filas)


cer_doc = error_por_item(salidas_ocr, refs_docs, voz.cer).merge(muestra_docs[["id", "tipo"]], on="id")
cer_doc.pivot_table(index=["tipo", "id"], columns="modelo", values="error").round(3)

modelo              PP-OCRv6 (base)  PaddleOCR-VL 1.6
tipo       id                                        
foto       doc_009            0.133             0.092
           doc_014            0.037             0.049
           doc_015            0.124             0.099
           doc_016            0.066             0.137
impreso    doc_001            0.022             0.063
           doc_007            0.033             0.014
           doc_008            0.047             0.043
manuscrito doc_022            0.137             0.140
           doc_023            0.487             0.210
           doc_025            0.492             0.382

## 5 · Extracción estructurada con un LLM

- **Modelo:** Qwen2.5-3B-Instruct (Alibaba, Apache 2.0), local, en la GPU de la Mac (bfloat16), con decodificación **greedy**
  (sin azar: el mismo texto siempre da la misma respuesta).
- **Entrada:** los textos OCR y ASR **ya seudonimizados** de los 25 documentos y los 28 clips. Los textos de más de 6000 caracteres se recortan
  (solo afecta a 2 portadas de periódico) para no exceder la memoria.
- **Salida:** un JSON con campos fijos. `modalidad`, `tema` y `resumen` siempre; en documentos `emisor`, `fecha`, `monto` y `lugar`
  **solo si aparecen en el texto**; en audios `citas_clave` (máximo 2 citas literales) y `numero_hablantes` (según las marcas `[HABLANTE_n]`).
  Lo que no aparece va como `null`: el prompt prohíbe inventar.
- Se guarda en `datos/extraccion_llm.csv` con `id`, `modalidad`, `campos_json`, `modelo_llm` y `fecha`.


In [7]:
from datetime import date

from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_LLM = "Qwen/Qwen2.5-3B-Instruct"
MAX_CARACTERES = 6000

INSTRUCCIONES = {
    "documento": '''Extrae información del siguiente texto obtenido por OCR de un documento peruano.
Responde SOLO con un objeto JSON válido, sin texto adicional, con estas claves:
- "modalidad": "documento"
- "tema": el asunto principal en pocas palabras
- "resumen": una o dos oraciones fieles al texto
- "emisor": quién emite o firma el documento, o null si no aparece
- "fecha": la fecha tal como aparece en el texto, o null si no aparece
- "monto": un monto de dinero que aparezca en el texto, o null si no aparece
- "lugar": el lugar mencionado como origen del documento, o null si no aparece
No inventes datos: si algo no está en el texto, usa null. El OCR puede tener errores; no los corrijas inventando.''',
    "audio": '''Extrae información de la siguiente transcripción automática de una entrevista en español peruano.
Las marcas [HABLANTE_1], [HABLANTE_2]... indican cambios de voz. Los nombres PERSONA_n son seudónimos.
Responde SOLO con un objeto JSON válido, sin texto adicional, con estas claves:
- "modalidad": "audio"
- "tema": el asunto principal en pocas palabras
- "resumen": una o dos oraciones fieles a la transcripción
- "citas_clave": una lista con hasta 2 frases copiadas literalmente de la transcripción, o null
- "numero_hablantes": el número de hablantes distintos según las marcas [HABLANTE_n], o 1 si no hay marcas
No inventes datos: si algo no está en el texto, usa null.''',
}

tok = llm = None


def cargar_llm():
    global tok, llm
    if llm is None:
        tok = AutoTokenizer.from_pretrained(MODELO_LLM)
        llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, dtype=torch.bfloat16).to(DISPOSITIVO).eval()


def extraer(modalidad, texto):
    "Pide el JSON al LLM y lo interpreta. Si la respuesta no es JSON válido, devuelve el error."
    cargar_llm()
    mensajes = [{"role": "system", "content": INSTRUCCIONES[modalidad]},
                {"role": "user", "content": texto[:MAX_CARACTERES]}]
    entrada = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(DISPOSITIVO)
    with torch.inference_mode():
        salida = llm.generate(**entrada, max_new_tokens=400, do_sample=False)
    respuesta = tok.decode(salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True)
    bloque = re.search(r"\{.*\}", respuesta, flags=re.S)
    try:
        return json.loads(bloque.group(0)), None
    except (AttributeError, json.JSONDecodeError) as e:
        return None, f"{type(e).__name__}: {respuesta[:200]}"


DIR_LLM = EVAL / "llm"
DIR_LLM.mkdir(exist_ok=True)
entradas = ([("documento", r.id, r.texto_ocr) for r in documentos.itertuples()]
            + [("audio", r.id, r.texto_asr) for r in transcripciones.itertuples()])
filas, t0 = [], time.perf_counter()
for modalidad, id_, texto in entradas:
    respaldo = DIR_LLM / f"{id_}.json"
    if respaldo.exists():
        r = json.loads(respaldo.read_text(encoding="utf-8"))
    else:
        campos, error = extraer(modalidad, texto)
        r = {"campos": campos, "error": error}
        respaldo.write_text(json.dumps(r, ensure_ascii=False, indent=1), encoding="utf-8")
    filas.append({"id": id_, "modalidad": modalidad, "campos_json": json.dumps(r["campos"], ensure_ascii=False) if r["campos"] else "",
                  "modelo_llm": f"{MODELO_LLM} (local, bf16, {DISPOSITIVO}, greedy)", "fecha": date.today().isoformat(),
                  "error": r["error"] or ""})
extraccion = pd.DataFrame(filas)
extraccion.drop(columns="error").to_csv(DATOS / "extraccion_llm.csv", index=False)
print(f"Extracciones: {len(extraccion)} | JSON inválidos: {(extraccion['error'] != '').sum()} | "
      f"tiempo: {(time.perf_counter() - t0) / 60:.1f} min")
extraccion.drop(columns="error").head()

Extracciones: 53 | JSON inválidos: 0 | tiempo: 0.0 min


,id,modalidad,campos_json,modelo_llm,fecha
0,doc_001,documento,"{""modalidad"": ""documento"", ""tema"": ""Muertes por COVID-19 en Perú"", ""resumen"": ""El Mini...","Qwen/Qwen2.5-3B-Instruct (local, bf16, mps, greedy)",2026-10-25
1,doc_002,documento,"{""modalidad"": ""documento"", ""tema"": ""Política y conflicto entre Ecuador y Perú"", ""resum...","Qwen/Qwen2.5-3B-Instruct (local, bf16, mps, greedy)",2026-10-25
2,doc_003,documento,"{""modalidad"": ""documento"", ""tema"": ""Noticia política y social"", ""resumen"": ""Se anuncia...","Qwen/Qwen2.5-3B-Instruct (local, bf16, mps, greedy)",2026-10-25
3,doc_004,documento,"{""modalidad"": ""documento"", ""tema"": ""Inundaciones en Huaraz"", ""resumen"": ""El documento ...","Qwen/Qwen2.5-3B-Instruct (local, bf16, mps, greedy)",2026-10-25
4,doc_005,documento,"{""modalidad"": ""documento"", ""tema"": ""conflicto choferes vs empresa"", ""resumen"": ""El tri...","Qwen/Qwen2.5-3B-Instruct (local, bf16, mps, greedy)",2026-10-25


### Auditoría humana de 20 extracciones

Se eligen **20 extracciones al azar** (10 documentos y 10 audios, semilla fija) y se revisan a mano comparándolas con el texto original.

**Criterio:** una extracción es **completamente correcta** si cumple todo esto a la vez:
1. es JSON válido y tiene todas las claves pedidas;
2. `tema` y `resumen` describen fielmente el texto, sin agregar hechos;
3. cada campo con valor (emisor, fecha, monto, lugar, citas) aparece de verdad en el texto; las citas son literales;
4. los campos que no aparecen en el texto están en `null` (no inventados);
5. `numero_hablantes` coincide con las marcas de la transcripción.

Un solo error en cualquier campo hace que la extracción cuente como incorrecta.
La hoja de auditoría se guarda en `datos/raw/auditoria_llm.csv`; la revisión se anota en las columnas `correcta` (sí/no) y `error_encontrado`.

In [8]:
HOJA = RAW / "auditoria_llm.csv"
if not HOJA.exists():
    muestra = pd.concat([extraccion[extraccion["modalidad"] == m].sample(10, random_state=2026) for m in ["documento", "audio"]])
    textos = pd.concat([documentos.set_index("id")["texto_ocr"], transcripciones.set_index("id")["texto_asr"]])
    hoja = muestra[["id", "modalidad", "campos_json"]].assign(texto=lambda d: d["id"].map(textos), correcta="", error_encontrado="")
    hoja.to_csv(HOJA, index=False)
auditoria = pd.read_csv(HOJA, keep_default_na=False)
revisadas = auditoria[auditoria["correcta"].str.strip() != ""]
print(f"Auditoría: {len(revisadas)}/20 extracciones revisadas")
if len(revisadas) == 20:
    correctas = auditoria["correcta"].str.strip().str.lower().isin(["sí", "si", "s"]).sum()
    print(f"Extracciones completamente correctas: {correctas}/20 = {correctas / 20:.0%}")
    display(auditoria.loc[~auditoria["correcta"].str.lower().isin(["sí", "si", "s"]), ["id", "modalidad", "error_encontrado"]])
auditoria[["id", "modalidad", "campos_json"]]

Auditoría: 0/20 extracciones revisadas


,id,modalidad,campos_json
0,doc_004,documento,"{""modalidad"": ""documento"", ""tema"": ""Inundaciones en Huaraz"", ""resumen"": ""El documento ..."
1,doc_016,documento,"{""modalidad"": ""documento"", ""tema"": ""historia y vida del general Juan Manuel de Iturrio..."
2,doc_019,documento,"{""modalidad"": ""documento"", ""tema"": ""asuntos consulares"", ""resumen"": ""El documento trat..."
3,doc_017,documento,"{""modalidad"": ""documento"", ""tema"": ""carta de César Vallejo a José María Eguren"", ""resu..."
4,doc_015,documento,"{""modalidad"": ""documento"", ""tema"": ""Falla de corteza terrestre en Colca"", ""resumen"": ""..."
5,doc_025,documento,"{""modalidad"": ""documento"", ""tema"": ""pagos y compras"", ""resumen"": ""Se hace referencia a..."
6,doc_005,documento,"{""modalidad"": ""documento"", ""tema"": ""conflicto choferes vs empresa"", ""resumen"": ""El tri..."
7,doc_009,documento,"{""modalidad"": ""documento"", ""tema"": ""Rehabilitación y remodelación del edificio Fenix"",..."
8,doc_011,documento,"{""modalidad"": ""documento"", ""tema"": ""Operaciones maritimas"", ""resumen"": ""Se describe un..."
9,doc_018,documento,"{""modalidad"": ""documento"", ""tema"": ""Denuncia y protesta contra actos violentos"", ""resu..."


## 6 · Comparativa, casos difíciles y decisión

### Tabla comparativa (muestra de 10 documentos y 10 clips)

In [9]:
ELEGIDO_OCR, ELEGIDO_ASR = "PaddleOCR-VL 1.6", "faster-whisper large-v3-turbo"
comp_ocr_tab = cer_tabla.assign(**{"segundos por documento": salidas_ocr.groupby("modelo")["segundos"].mean().round(2)})
comp_asr_tab = wer_tabla.join(latencia["factor_tiempo_real"])
print("OCR – CER por tipo y tiempo:"); display(comp_ocr_tab)
print("ASR – WER por condición y factor de tiempo real:"); display(comp_asr_tab)
print(f"Resultados de los modelos elegidos → CER ({ELEGIDO_OCR}): {cer_tabla.loc[ELEGIDO_OCR, 'GLOBAL']:.2f} | "
      f"WER ({ELEGIDO_ASR}): {wer_tabla.loc[ELEGIDO_ASR, 'GLOBAL']:.2f}")

OCR – CER por tipo y tiempo:


tipo,GLOBAL,foto,impreso,manuscrito,segundos por documento
modelo,,,,,
PP-OCRv6 (base),0.111,0.101,0.036,0.364,3.04
PaddleOCR-VL 1.6,0.096,0.104,0.041,0.258,5.46


ASR – WER por condición y factor de tiempo real:


,GLOBAL,calle,feria,fiesta patronal,lectura limpia,factor_tiempo_real
modelo,,,,,,
Qwen3-ASR 0.6B,0.302,0.197,0.320,0.361,0.000,0.096
faster-whisper large-v3-turbo,0.236,0.184,0.338,0.236,0.000,0.270
faster-whisper small (base),0.349,0.262,0.316,0.421,0.043,0.149


Resultados de los modelos elegidos → CER (PaddleOCR-VL 1.6): 0.10 | WER (faster-whisper large-v3-turbo): 0.24


### Casos difíciles y reflexión por tipo

**Documentos.** Los **impresos** son el mejor caso (~4 % de CER con ambos modelos): tipografía regular y buen contraste.
Las **fotos** quedan en ~10 %: la perspectiva, los reflejos del bronce y la "V" romana de las placas (el OCR copia "FVE" bien, pero confunde letras talladas)
generan errores parecidos en los dos modelos. Los **manuscritos** son el peor caso: PP-OCRv6 está entrenado con texto impreso y separa mal la cursiva
(36 %); PaddleOCR-VL, que genera el texto como un modelo de lenguaje, aprovecha el contexto y baja a 26 %, aunque el oficio más antiguo y abreviado sigue
casi ilegible para ambos.

**Audio.** La **lectura limpia** de OpenSLR es casi perfecta (0–4 %). En la **calle** el error sube a ~18–26 % por el ruido de tráfico y las voces superpuestas.
La **fiesta patronal** es el caso más difícil: audio saturado y música de fondo, donde el VAD falla (se aplicó la regla de respaldo) y el modelo pequeño
llega a 42 %. En la **feria**, los tres modelos quedan en ~32–34 %: hay voces de fondo y frases cortadas que ningún modelo recupera.
El tamaño del modelo importa más que la arquitectura: Whisper *turbo* (0,8 B) supera a *small* (0,24 B) en todas las condiciones con ruido.

### Decisión para un entorno real
**OCR → PaddleOCR-VL 1.6.** Menor CER global (9,6 % frente a 11,1 %) y, sobre todo, en manuscritos (25,8 % frente a 36,4 %), que es donde está el problema real del corpus. En impresos y fotos ambos empatan (~4 % y ~10 %). Cuesta 1,8 veces más tiempo (5,5 s frente a 3,0 s por página), aceptable para un archivo que se procesa una sola vez. Si el volumen fuera enorme y solo de impresos, PP-OCRv6 sería suficiente y más barato.

**ASR → faster-whisper large-v3-turbo.** El menor WER global (23,6 %, frente a 34,9 % de la línea base *small* y 30,2 % de Qwen3-ASR) y el mejor en el caso más difícil, la fiesta patronal con ruido (23,6 % frente a 42,1 % y 36,1 %). Corre en CPU sin GPU (factor de tiempo real 0,27: una hora de audio en ~16 min). Qwen3-ASR es el más rápido (0,10) y empata en calle, así que sería la opción si hubiera que transcribir en tiempo real con GPU.

## 7 · Artefactos ligeros para el dashboard

El dashboard no ejecuta modelos: lee estos archivos de `modelos/`.
- `metricas.json`: CER y WER por modelo y tipo/condición, latencia, acuerdo LLM–humano y la decisión final.
- `comparacion_ocr.csv` y `comparacion_asr.csv`: el texto de cada modelo en cada documento o clip de la muestra, con su error
  (las transcripciones de audio, seudonimizadas).

In [10]:
DECISION = '**OCR → PaddleOCR-VL 1.6.** Menor CER global (9,6 % frente a 11,1 %) y, sobre todo, en manuscritos (25,8 % frente a 36,4 %), que es donde está el problema real del corpus. En impresos y fotos ambos empatan (~4 % y ~10 %). Cuesta 1,8 veces más tiempo (5,5 s frente a 3,0 s por página), aceptable para un archivo que se procesa una sola vez. Si el volumen fuera enorme y solo de impresos, PP-OCRv6 sería suficiente y más barato.\n\n**ASR → faster-whisper large-v3-turbo.** El menor WER global (23,6 %, frente a 34,9 % de la línea base *small* y 30,2 % de Qwen3-ASR) y el mejor en el caso más difícil, la fiesta patronal con ruido (23,6 % frente a 42,1 % y 36,1 %). Corre en CPU sin GPU (factor de tiempo real 0,27: una hora de audio en ~16 min). Qwen3-ASR es el más rápido (0,10) y empata en calle, así que sería la opción si hubiera que transcribir en tiempo real con GPU.'
MODELOS_DIR = Path("modelos")
cer_doc.merge(salidas_ocr[["id", "modelo", "texto", "segundos"]], on=["id", "modelo"]).rename(columns={"error": "cer"})[
    ["id", "tipo", "modelo", "cer", "segundos", "texto"]].to_csv(MODELOS_DIR / "comparacion_ocr.csv", index=False)
wer_clip = error_por_item(salidas_asr, refs_clips, voz.wer)
(wer_clip.merge(salidas_asr[["id", "modelo", "texto", "segundos"]], on=["id", "modelo"]).rename(columns={"error": "wer"})
 .assign(texto=lambda d: d["texto"].map(seudonimizar))[["id", "modelo", "wer", "segundos", "texto"]]
 .to_csv(MODELOS_DIR / "comparacion_asr.csv", index=False))

a_dict = lambda t: {m: {k: (None if pd.isna(v) else float(v)) for k, v in fila.items()} for m, fila in t.iterrows()} if t is not None else {}
correctas = int(auditoria["correcta"].str.strip().str.lower().isin(["sí", "si", "s"]).sum()) if len(revisadas) == 20 else None
resumen = {
    "cer": a_dict(cer_tabla), "wer": a_dict(wer_tabla),
    "latencia_asr": latencia["factor_tiempo_real"].to_dict(),
    "segundos_por_documento": salidas_ocr.groupby("modelo")["segundos"].mean().round(2).to_dict(),
    "acuerdo_llm": {"correctas": correctas, "total": 20} if correctas is not None else {},
    "modelo_ocr_elegido": globals().get("ELEGIDO_OCR"), "modelo_asr_elegido": globals().get("ELEGIDO_ASR"),
    "decision": globals().get("DECISION", ""),
}
(MODELOS_DIR / "metricas.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps({k: v for k, v in resumen.items() if k != "decision"}, ensure_ascii=False, indent=1)[:1500])

{
 "cer": {
  "PP-OCRv6 (base)": {
   "GLOBAL": 0.111,
   "foto": 0.101,
   "impreso": 0.036,
   "manuscrito": 0.364
  },
  "PaddleOCR-VL 1.6": {
   "GLOBAL": 0.096,
   "foto": 0.104,
   "impreso": 0.041,
   "manuscrito": 0.258
  }
 },
 "wer": {
  "Qwen3-ASR 0.6B": {
   "GLOBAL": 0.302,
   "calle": 0.197,
   "feria": 0.32,
   "fiesta patronal": 0.361,
   "lectura limpia": 0.0
  },
  "faster-whisper large-v3-turbo": {
   "GLOBAL": 0.236,
   "calle": 0.184,
   "feria": 0.338,
   "fiesta patronal": 0.236,
   "lectura limpia": 0.0
  },
  "faster-whisper small (base)": {
   "GLOBAL": 0.349,
   "calle": 0.262,
   "feria": 0.316,
   "fiesta patronal": 0.421,
   "lectura limpia": 0.043
  }
 },
 "latencia_asr": {
  "Qwen3-ASR 0.6B": 0.096,
  "faster-whisper large-v3-turbo": 0.27,
  "faster-whisper small (base)": 0.149
 },
 "segundos_por_documento": {
  "PP-OCRv6 (base)": 3.04,
  "PaddleOCR-VL 1.6": 5.46
 },
 "acuerdo_llm": {},
 "modelo_ocr_elegido": "PaddleOCR-VL 1.6",
 "modelo_asr_elegido": "f